# EPCLM-005 H4 L1-C contextual memory screening
Select Tesla T4, enable Internet and the existing GITHUB_TOKEN, then Save & Run All. CPU and T4 gates precede the bounded four-arm run. Attach complete saved Output to resume or retry publication.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import signal
import subprocess
import sys
import time

started=time.monotonic()
stop=started+10500
remote='https://github.com/ArcheLabs/mini-cells.git'
branch='research/epclm005-h4-l1c'
repo=Path('/kaggle/working/minicells-epclm005-h4-l1c')
os.environ['OMP_NUM_THREADS']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'

def run(argv,limit):
    remaining=min(limit,stop-time.monotonic())
    if remaining<=0:
        raise TimeoutError('H4 session budget exhausted')
    process=subprocess.Popen(argv,start_new_session=True)
    try:
        code=process.wait(timeout=remaining)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid,signal.SIGTERM)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL)
            process.wait()
        raise TimeoutError('H4 L1-C stopped at session budget. Preserve complete Output.')
    if code:
        raise RuntimeError('H4 L1-C stopped. Inspect the retained error and checkpoints.')

if not repo.exists():
    run(['git','clone','--depth=1','--branch',branch,remote,str(repo)],300)
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
base=repo/'research/projects/cellular-language-models/experiments/epclm-004-unified-memory-10m/m4'
out=Path('/kaggle/working/epclm005-h4-l1c-'+commit[:12])
run([sys.executable,'-m','pip','install','-q','-r',str(base/'requirements.txt')],300)
if not out.exists():
    candidates=[]
    for f in Path('/kaggle/input').rglob('execution-identity.json'):
        identity=json.loads(f.read_text())
        if f.parent.name==out.name and identity.get('source_commit')==commit:
            complete=(f.parent/'comparison.json').exists()
            candidates.append((complete,sum((f.parent/arm/'metrics.json').is_file() for arm in ('F00','F10','F01','F11')),f))
    if candidates:
        previous=max(candidates,key=lambda item:(item[0],item[1],str(item[2])))[2]
        shutil.copytree(previous.parent,out)
os.chdir(repo)
sys.path.insert(0,str(repo))
from scripts.research.publish_experiment_results import load_github_token
load_github_token('GITHUB_TOKEN')
run([sys.executable,str(base/'l1c/l1c.py'),'--output',str(out),
     '--started',str(started)],10500)
